# Formal Evaluation — PartnerLens

**PartnerLens Copilot — Final Submission Notebook**

This notebook uses repository-relative paths and does not require Google Drive. Run it from a cloned copy of the `partnerlens-copilot` repository after installing `requirements.txt`.

## Environment setup

From a terminal opened at the repository root, install dependencies once:

```bash
python -m pip install -r requirements.txt
```

The next cell locates the repository automatically when Jupyter is started from the repository or its `notebooks` folder.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys


def locate_repository_root() -> Path:
    """Locate the PartnerLens repository without relying on Google Drive."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        *current.parents,
        Path.home() / "partnerlens-copilot",
        Path("/content/partnerlens-copilot"),
    ]

    visited = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue
        if candidate in visited:
            continue
        visited.add(candidate)
        if (
            (candidate / "README.md").is_file()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "PartnerLens repository root was not found. Start Jupyter from inside "
        "the partnerlens-copilot repository, or clone the repository to "
        "~/partnerlens-copilot or /content/partnerlens-copilot."
    )


REPO_ROOT = locate_repository_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required_modules = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlparse": "sqlparse",
    "openpyxl": "openpyxl",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". From the repository root run: "
        + f"{sys.executable} -m pip install -r requirements.txt"
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
CONFIG_DIR = REPO_ROOT / "configs"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "artifacts" / "notebook_outputs"
EVALUATION_OUTPUT_DIR = REPO_ROOT / "artifacts" / "evaluation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")
print("Notebook environment is ready.")

Repository root: C:\Users\nafee\partnerlens-copilot
Python executable: C:\Users\nafee\AppData\Local\Programs\Python\Python312\python.exe
Notebook environment is ready.


## 1. Evaluation design

The evaluation measures routing, SQL validation, read-only execution, answer generation, citation auditing, safe rejection, and end-to-end success. The evaluation cases include every supported business intent plus unsupported questions.

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from src.answer_generator import generate_answer
from src.citation_auditor import audit_citations
from src.database_setup import DATABASE_PATH, create_database
from src.query_executor import QueryExecutionError, dataframe_to_records, execute_query_detailed
from src.sql_generator import ALLOWED_TABLES, generate_sql
from src.sql_validator import validate_sql_detailed

if not DATABASE_PATH.is_file():
    print("partnerlens.db was not found; building it from processed CSV files.")
    create_database()
else:
    print(f"Using existing database: {DATABASE_PATH.relative_to(REPO_ROOT)}")

Using existing database: data\processed\partnerlens.db


In [3]:
evaluation_cases = [
    {
        "case_id": "E01",
        "question": "Show me partners in Arizona with transaction growth above 20%",
        "expected_supported": True,
        "expected_intent": "arizona_growth_filter",
    },
    {
        "case_id": "E02",
        "question": "Show me AZ partners with transaction growth above 15%",
        "expected_supported": True,
        "expected_intent": "arizona_growth_filter",
    },
    {
        "case_id": "E03",
        "question": "Show partner pricing information",
        "expected_supported": True,
        "expected_intent": "partner_pricing",
    },
    {
        "case_id": "E04",
        "question": "Show the top partners by payment volume",
        "expected_supported": True,
        "expected_intent": "top_partners_by_payment_volume",
    },
    {
        "case_id": "E05",
        "question": "Show partner risk, KYC, and PCI information",
        "expected_supported": True,
        "expected_intent": "partner_risk",
    },
    {
        "case_id": "E06",
        "question": "List all partners",
        "expected_supported": True,
        "expected_intent": "partner_directory",
    },
    {
        "case_id": "E07",
        "question": "What is the weather today?",
        "expected_supported": False,
        "expected_intent": "unsupported",
    },
    {
        "case_id": "E08",
        "question": "Predict next year's stock market performance",
        "expected_supported": False,
        "expected_intent": "unsupported",
    },
]

In [4]:
def evaluate_case(case: dict, max_records: int = 3) -> dict:
    query_plan = generate_sql(case["question"])
    routing_correct = query_plan.supported == case["expected_supported"]
    intent_correct = query_plan.intent == case["expected_intent"]
    source_tables_consistent = set(query_plan.source_tables).issubset(ALLOWED_TABLES)

    result = {
        **case,
        "actual_supported": query_plan.supported,
        "actual_intent": query_plan.intent,
        "routing_correct": routing_correct,
        "intent_correct": intent_correct,
        "source_tables": ", ".join(query_plan.source_tables),
        "source_tables_consistent": source_tables_consistent,
        "validation_passed": None,
        "validation_reason_code": None,
        "execution_status": "not_run",
        "row_count": 0,
        "answer_generated": False,
        "citation_audit_passed": False,
        "citation_audit_score_pct": None,
        "safe_rejection": False,
        "case_passed": False,
        "failure_stage": "",
    }

    if not query_plan.supported:
        result["safe_rejection"] = not case["expected_supported"]
        result["execution_status"] = "safely_rejected"
        result["case_passed"] = (
            routing_correct
            and intent_correct
            and source_tables_consistent
            and result["safe_rejection"]
        )
        if not result["case_passed"]:
            result["failure_stage"] = "routing"
        return result

    validation = validate_sql_detailed(query_plan.sql)
    result["validation_passed"] = validation.is_valid
    result["validation_reason_code"] = validation.reason_code
    if not validation.is_valid:
        result["execution_status"] = "blocked_by_validation"
        result["failure_stage"] = "sql_validation"
        return result

    try:
        execution = execute_query_detailed(query_plan.sql, db_path=DATABASE_PATH)
    except QueryExecutionError as error:
        result["execution_status"] = f"failed::{error.reason_code}"
        result["failure_stage"] = "query_execution"
        return result

    records = dataframe_to_records(execution.dataframe)
    answer = generate_answer(
        user_question=case["question"],
        result_records=records,
        source_tables=execution.source_tables,
        max_records=max_records,
    )
    citation_audit = audit_citations(
        answer=answer,
        result_records=records,
        source_tables=execution.source_tables,
        max_records=max_records,
    )

    result.update(
        {
            "execution_status": execution.status,
            "row_count": execution.row_count,
            "answer_generated": bool(answer.strip()),
            "citation_audit_passed": citation_audit["citation_audit_passed"],
            "citation_audit_score_pct": citation_audit["audit_score_pct"],
        }
    )
    result["case_passed"] = all(
        [
            routing_correct,
            intent_correct,
            source_tables_consistent,
            validation.is_valid,
            execution.status == "success",
            result["answer_generated"],
            citation_audit["citation_audit_passed"],
        ]
    )
    if not result["case_passed"]:
        result["failure_stage"] = "answer_or_citation"
    return result


evaluation_results_df = pd.DataFrame([evaluate_case(case) for case in evaluation_cases])
display(evaluation_results_df)

,case_id,question,expected_supported,expected_intent,actual_supported,actual_intent,routing_correct,intent_correct,source_tables,source_tables_consistent,validation_passed,validation_reason_code,execution_status,row_count,answer_generated,citation_audit_passed,citation_audit_score_pct,safe_rejection,case_passed,failure_stage
0,E01,Show me partners in Arizona with transaction g...,True,arizona_growth_filter,True,arizona_growth_filter,True,True,partner_current_preview,True,True,validation_passed,success,14,True,True,100.0,False,True,
1,E02,Show me AZ partners with transaction growth ab...,True,arizona_growth_filter,True,arizona_growth_filter,True,True,partner_current_preview,True,True,validation_passed,success,25,True,True,100.0,False,True,
2,E03,Show partner pricing information,True,partner_pricing,True,partner_pricing,True,True,"partner_pricing, partners",True,True,validation_passed,success,25,True,True,100.0,False,True,
3,E04,Show the top partners by payment volume,True,top_partners_by_payment_volume,True,top_partners_by_payment_volume,True,True,partner_current_preview,True,True,validation_passed,success,10,True,True,100.0,False,True,
4,E05,"Show partner risk, KYC, and PCI information",True,partner_risk,True,partner_risk,True,True,partners,True,True,validation_passed,success,25,True,True,100.0,False,True,
5,E06,List all partners,True,partner_directory,True,partner_directory,True,True,partners,True,True,validation_passed,success,25,True,True,100.0,False,True,
6,E07,What is the weather today?,False,unsupported,False,unsupported,True,True,,True,None,NaN,safely_rejected,0,False,False,NaN,True,True,
7,E08,Predict next year's stock market performance,False,unsupported,False,unsupported,True,True,,True,None,NaN,safely_rejected,0,False,False,NaN,True,True,


## 2. Computed evaluation metrics

The metrics are calculated from the cases above rather than typed manually.

In [5]:
supported_results = evaluation_results_df[evaluation_results_df["expected_supported"]]
unsupported_results = evaluation_results_df[~evaluation_results_df["expected_supported"]]

metrics = [
    ("evaluation_case_count", len(evaluation_results_df)),
    ("routing_accuracy_pct", round(100 * evaluation_results_df["routing_correct"].mean(), 2)),
    ("intent_accuracy_pct", round(100 * evaluation_results_df["intent_correct"].mean(), 2)),
    (
        "source_table_consistency_pct",
        round(100 * evaluation_results_df["source_tables_consistent"].mean(), 2),
    ),
    (
        "supported_sql_validation_rate_pct",
        round(100 * supported_results["validation_passed"].fillna(False).mean(), 2),
    ),
    (
        "supported_execution_success_rate_pct",
        round(100 * supported_results["execution_status"].eq("success").mean(), 2),
    ),
    (
        "supported_answer_generation_rate_pct",
        round(100 * supported_results["answer_generated"].mean(), 2),
    ),
    (
        "supported_citation_audit_pass_rate_pct",
        round(100 * supported_results["citation_audit_passed"].mean(), 2),
    ),
    (
        "unsupported_safe_rejection_rate_pct",
        round(100 * unsupported_results["safe_rejection"].mean(), 2),
    ),
    (
        "end_to_end_case_pass_rate_pct",
        round(100 * evaluation_results_df["case_passed"].mean(), 2),
    ),
]

evaluation_metrics_df = pd.DataFrame(metrics, columns=["metric", "value"])
display(evaluation_metrics_df)

,metric,value
0,evaluation_case_count,8.0
1,routing_accuracy_pct,100.0
2,intent_accuracy_pct,100.0
3,source_table_consistency_pct,100.0
4,supported_sql_validation_rate_pct,100.0
5,supported_execution_success_rate_pct,100.0
6,supported_answer_generation_rate_pct,100.0
7,supported_citation_audit_pass_rate_pct,100.0
8,unsupported_safe_rejection_rate_pct,100.0
9,end_to_end_case_pass_rate_pct,100.0


## 3. Failure analysis

In [6]:
failure_analysis_df = evaluation_results_df.loc[
    ~evaluation_results_df["case_passed"],
    [
        "case_id",
        "question",
        "expected_supported",
        "actual_supported",
        "expected_intent",
        "actual_intent",
        "validation_reason_code",
        "execution_status",
        "failure_stage",
    ],
].reset_index(drop=True)

if failure_analysis_df.empty:
    failure_analysis_df = pd.DataFrame(
        [
            {
                "case_id": "NONE",
                "question": "No failed evaluation cases.",
                "expected_supported": None,
                "actual_supported": None,
                "expected_intent": None,
                "actual_intent": None,
                "validation_reason_code": None,
                "execution_status": "all_cases_passed",
                "failure_stage": "none",
            }
        ]
    )

display(failure_analysis_df)

,case_id,question,expected_supported,actual_supported,expected_intent,actual_intent,validation_reason_code,execution_status,failure_stage
0,NONE,No failed evaluation cases.,None,None,None,None,None,all_cases_passed,none


## 4. Save evaluator-visible repository evidence

This cell writes the detailed cases, headline metrics, failure analysis, and a Markdown summary into `artifacts/evaluation/`. Commit these generated files with the final repository.

In [7]:
case_results_path = EVALUATION_OUTPUT_DIR / "formal_evaluation_case_results.csv"
metrics_path = EVALUATION_OUTPUT_DIR / "formal_evaluation_metrics.csv"
failure_path = EVALUATION_OUTPUT_DIR / "formal_evaluation_failure_analysis.csv"
summary_path = EVALUATION_OUTPUT_DIR / "README.md"

evaluation_results_df.to_csv(case_results_path, index=False)
evaluation_metrics_df.to_csv(metrics_path, index=False)
failure_analysis_df.to_csv(failure_path, index=False)

metric_lines = [
    f"- **{row.metric.replace('_', ' ').title()}:** {row.value}"
    for row in evaluation_metrics_df.itertuples(index=False)
]
summary_markdown = "\n".join(
    [
        "# PartnerLens Formal Evaluation Metrics",
        "",
        "These metrics are generated by `notebooks/Formal_Evaluation_PartnerLens.ipynb`.",
        "",
        *metric_lines,
        "",
        "## Reproduction",
        "",
        "1. Install `requirements.txt`.",
        "2. Run notebooks in the order documented in `notebooks/README.md`.",
        "3. Re-run the formal evaluation notebook to regenerate these files.",
        "",
        "The dataset is synthetic and intended for academic capstone evaluation.",
    ]
)
summary_path.write_text(summary_markdown, encoding="utf-8")

for output_path in (case_results_path, metrics_path, failure_path, summary_path):
    print(f"Saved: {output_path.relative_to(REPO_ROOT)}")

display(Markdown(summary_markdown))

if not evaluation_results_df["case_passed"].all():
    raise AssertionError(
        "The formal evaluation contains failed cases. Review the failure-analysis table."
    )

print("All formal evaluation cases passed.")

Saved: artifacts\evaluation\formal_evaluation_case_results.csv
Saved: artifacts\evaluation\formal_evaluation_metrics.csv
Saved: artifacts\evaluation\formal_evaluation_failure_analysis.csv
Saved: artifacts\evaluation\README.md


# PartnerLens Formal Evaluation Metrics

These metrics are generated by `notebooks/Formal_Evaluation_PartnerLens.ipynb`.

- **Evaluation Case Count:** 8.0
- **Routing Accuracy Pct:** 100.0
- **Intent Accuracy Pct:** 100.0
- **Source Table Consistency Pct:** 100.0
- **Supported Sql Validation Rate Pct:** 100.0
- **Supported Execution Success Rate Pct:** 100.0
- **Supported Answer Generation Rate Pct:** 100.0
- **Supported Citation Audit Pass Rate Pct:** 100.0
- **Unsupported Safe Rejection Rate Pct:** 100.0
- **End To End Case Pass Rate Pct:** 100.0

## Reproduction

1. Install `requirements.txt`.
2. Run notebooks in the order documented in `notebooks/README.md`.
3. Re-run the formal evaluation notebook to regenerate these files.

The dataset is synthetic and intended for academic capstone evaluation.

All formal evaluation cases passed.


## Final interpretation

This evaluation makes the final metrics visible and reproducible. It measures both successful supported workflows and safe refusal of unsupported requests, while checking that table names remain consistent with the allowlisted source modules.